In [1]:
import time

import numpy as np
import pandas as pd
import torch
import yaml
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from torch import nn

torch.manual_seed(42)
np.random.seed(42)

In [2]:
# load data
water_value = pd.read_csv("data/kernel/Synthetic_water_value_2015_2024.csv",
    parse_dates=["date"],
    index_col="date"
)

volume = pd.read_csv("data/kernel/Historical_volume_2015_2024.csv",
    parse_dates=["date"],
    index_col="date"
)

day_ahead_price = pd.read_csv("data/kernel/Historical_day_ahead_price_2015_2025.csv",
    parse_dates=["date"],
    index_col="date"
)

# rename price column and resample to hourly for the 15 minute interval period
day_ahead_price = day_ahead_price.rename(columns={"Day-ahead price (EUR/MWh)": "price"})
day_ahead_price = day_ahead_price.resample("h").mean()

inflow = pd.read_csv("data/kernel/Historical_inflow_1958_2025.csv",
    parse_dates=["date"],
    index_col="date"
)

# Unit commitment decisions have no date column, so we use starttime instead
unit_commitment_decisions = pd.read_csv("data/kernel/Unit_commitment_decisions.csv",
    parse_dates=["starttime"],
    index_col="starttime"
)

# drop the timezone for consistency
water_value.index = water_value.index.tz_convert(None)
volume.index = volume.index.tz_convert(None)
day_ahead_price.index = day_ahead_price.index.tz_convert(None)
inflow.index = inflow.index.tz_convert(None)

# reservoir fill level as a share of max volume (max_vol from the topology file)
with open("data/extended/Tokke_Vinje_topology.yaml") as f:
    topology = yaml.safe_load(f)

max_vol = pd.Series({name: r["max_vol"] for name, r in topology["model"]["reservoir"].items()})
fill = volume / max_vol

# test cases (2023–2024) to predict for Kaggle
test_cases = pd.read_csv("prediction_mapping.csv", parse_dates=["starttime"])

# Max flow (m³/s) each unit can use at full load = the largest flow value in its turbine efficiency curves
unit_max_flow = {gen: max(max(curve["x"]) for curve in g["turb_eff_curves"])
                 for gen, g in topology["model"]["generator"].items()}

plant_max_flow = {}
for gen, q in unit_max_flow.items():
    plant = gen.rsplit("_G", 1)[0]
    plant_max_flow[plant] = plant_max_flow.get(plant, 0) + q   # plant total = sum of its units

plant_max_flow

{'Hogga': 168.6,
 'Tokke': 128,
 'Lio': 14,
 'Byrte': 8,
 'Vinje': 165,
 'Haukeli': 2.1,
 'Songa': 52,
 'Kjela': 40,
 'Vesle Kjela': 20}

In [3]:
#Find the units in column names
label_cols = [c for c in unit_commitment_decisions.columns if c.startswith("result_committed")]

units = []
for c in label_cols:
    unit = c.replace("result_committed_", "").rsplit("_t", 1)[0] # making the names: "Hogga_G1_t5" -> "Hogga_G1"
    if unit not in units:
        units.append(unit)

print(len(units), units)

# The columns come in order: all 168 hours for unit 1, then all 168 for unit 2, up to 14
Y = unit_commitment_decisions[label_cols].to_numpy().reshape(len(unit_commitment_decisions), len(units), 168)
Y.shape

14 ['Hogga_G1', 'Tokke_G1', 'Tokke_G2', 'Tokke_G3', 'Tokke_G4', 'Lio_G1', 'Byrte_G1', 'Vinje_G1', 'Vinje_G2', 'Vinje_G3', 'Haukeli_G1', 'Songa_G1', 'Kjela_G1', 'Vesle Kjela_G1']


(2922, 14, 168)

In [4]:
assert (Y[0, 0, :] == unit_commitment_decisions.filter(like="Hogga_G1_t").iloc[0].to_numpy()).all()
assert (Y[5, 13, :] == unit_commitment_decisions.filter(like="Vesle Kjela_G1_t").iloc[5].to_numpy()).all()
print("Reshape OK, labels only 0/1:", set(pd.unique(Y.ravel())))

Reshape OK, labels only 0/1: {np.int64(0), np.int64(1)}


In [5]:
# Find which reservoir(s) each plant draws water from, using the topology file
plant_sources = {}
for conn in topology["connections"]:
    if conn["to_type"] == "plant" and conn["from_type"] in ("reservoir", "tunnel"):
        plant_sources.setdefault(conn["to"], []).append(conn["from"])

# Some plants get water through a tunnel: replace the tunnel with the reservoir it starts at
tunnel_sources = {conn["to"]: conn["from"] for conn in topology["connections"]
                  if conn["to_type"] == "tunnel" and conn["from_type"] == "reservoir"}
plant_reservoir = {plant: [tunnel_sources.get(s, s) for s in sources]
                   for plant, sources in plant_sources.items()}

for plant, reservoirs in plant_reservoir.items():
    print(f"{plant:12s} <- {reservoirs}")

Hogga        <- ['Bandak']
Tokke        <- ['Vinjevatn']
Lio          <- ['Byrtevatn']
Byrte        <- ['Botnedalsvatn']
Vinje        <- ['Vaamarvatn']
Haukeli      <- ['Vatjern']
Songa        <- ['Songavatn', 'Bitdalsvatn']
Kjela        <- ['Foersvatn']
Vesle Kjela  <- ['Kjelavatn']


In [6]:
# Which reservoir each plant releases its water into (downstream)
plant_downstream = {}
for conn in topology["connections"]:
    if conn["from_type"] == "plant" and conn["to_type"] == "reservoir":
        plant_downstream[conn["from"]] = [conn["to"]]
plant_downstream["Lio"] = ["Bandak"]   # Lio -> river r_Vest_Vassdraget -> Bandak
plant_downstream["Hogga"] = []         # Hogga is at the bottom: the water leaves the system

for plant in plant_reservoir:
    print(f"{plant:12s} {plant_reservoir[plant]} -> {plant_downstream[plant]}")

Hogga        ['Bandak'] -> []
Tokke        ['Vinjevatn'] -> ['Bandak']
Lio          ['Byrtevatn'] -> ['Bandak']
Byrte        ['Botnedalsvatn'] -> ['Byrtevatn']
Vinje        ['Vaamarvatn'] -> ['Vinjevatn']
Haukeli      ['Vatjern'] -> ['Vinjevatn']
Songa        ['Songavatn', 'Bitdalsvatn'] -> ['Totak']
Kjela        ['Foersvatn'] -> ['Hyljelihyl']
Vesle Kjela  ['Kjelavatn'] -> ['Foersvatn']


In [7]:
# Which plants release water into the reservoir(s) this plant draws from
plant_upstream = {plant: [other for other, down in plant_downstream.items()
                          if set(down) & set(plant_reservoir[plant])]
                  for plant in plant_reservoir}
plant_upstream["Vinje"] = ["Songa"]   # Songa -> Totak -> tunnel -> Vaamarvatn (not a direct connection)
plant_upstream

{'Hogga': ['Tokke', 'Lio'],
 'Tokke': ['Vinje', 'Haukeli'],
 'Lio': ['Byrte'],
 'Byrte': [],
 'Vinje': ['Songa'],
 'Haukeli': [],
 'Songa': [],
 'Kjela': ['Vesle Kjela'],
 'Vesle Kjela': []}

In [8]:
# "Normal" inflow for each day of the year, from 1958–2014
inflow_daily = inflow.resample("D").mean()
inflow_normal = inflow_daily.loc[:"2014"].groupby(inflow_daily.loc[:"2014"].index.dayofyear).mean()

def hourly_matrix(series, starts):
    """168 hourly values from each start date -> shape (n_cases, 168)."""
    return np.array([series.loc[s : s + pd.Timedelta(hours=167)].to_numpy() for s in starts])

def rolling_within_case(M, window):
    """Centred rolling mean along the hours of each case."""
    return pd.DataFrame(M).T.rolling(window, center=True, min_periods=1).mean().T.to_numpy()

def daily_value(df, dates, columns):
    """Mean of the given reservoir columns on the given dates (0 if there are no reservoirs)."""
    if len(columns) == 0:
        return np.zeros(len(dates))
    return df.loc[dates, columns].mean(axis=1).to_numpy()
    

In [9]:
def build_features(starts):
    """One row per (case, unit, hour). Works for both train and test start dates."""
    starts = pd.DatetimeIndex(starts)
    ends = starts + pd.Timedelta(days=7)
    n_cases, n_hours = len(starts), 168
    rep = lambda v: np.repeat(v, n_hours)            # one value per case -> all 168 hours
    doy = np.minimum(starts.dayofyear, 365)

    # ---------- features that are the same for all units ----------
    P = hourly_matrix(day_ahead_price["price"], starts)
    desc_rank = n_hours - P.argsort(axis=1).argsort(axis=1)   # 1 = most expensive hour of the week, 168 = cheapest
    week_mean = P.mean(axis=1, keepdims=True)
    week_std = P.std(axis=1, keepdims=True) + 1e-6
    total_inflow = hourly_matrix(inflow.sum(axis=1), starts)
    total_normal = inflow_normal.sum(axis=1).loc[doy].to_numpy()

    shared = {
        "hour": np.tile(np.arange(n_hours), n_cases),
        "hour_of_day": np.tile(np.arange(n_hours) % 24, n_cases),
        "month": rep(starts.month.to_numpy()),
        "price": P.ravel(),
        "price_rank": (P.argsort(axis=1).argsort(axis=1) / (n_hours - 1)).ravel(),
        "price_z": ((P - week_mean) / week_std).ravel(),
        "price_week_mean": rep(week_mean[:, 0]),
        "price_roll6": rolling_within_case(P, 7).ravel(),
        "price_roll24": rolling_within_case(P, 25).ravel(),
        "inflow_total_week": rep(total_inflow.mean(axis=1)),
        "inflow_total_anom": rep(total_inflow.mean(axis=1) / total_normal),
    }

    # price minus end water value for every plant (used for the cascade features)
    diff_end_by_plant = {pl: P - daily_value(water_value, ends, plant_reservoir[pl])[:, None]
                         for pl in plant_reservoir}

    # ---------- features that depend on the unit's reservoirs ----------
    blocks = []
    for unit in units:
        plant = unit.rsplit("_G", 1)[0]
        up = plant_reservoir[plant]          # where the water comes from
        down = plant_downstream[plant]       # where the water goes

        wv_start = daily_value(water_value, starts, up)
        wv_end = daily_value(water_value, ends, up)                         # THE key quantity
        net_wv_end = wv_end - daily_value(water_value, ends, down)          # value lost by moving water down
        diff_end = P - wv_end[:, None]                                      # price minus end water value
        diff_net = P - net_wv_end[:, None]
        res_inflow = hourly_matrix(inflow[up].sum(axis=1), starts)
        res_normal = inflow_normal[up].sum(axis=1).loc[doy].to_numpy()

        # ---------- water budget ----------
        vol_start = volume.loc[starts, up].sum(axis=1).to_numpy()             # Mm³ in the reservoir(s) at start
        inflow_week = res_inflow.sum(axis=1) * 3600 / 1e6                     # Mm³ flowing in during the week
        free_space = max_vol[up].sum() - vol_start                            # Mm³ before the reservoir is full
        plant_per_hour = plant_max_flow[plant] * 3600 / 1e6                   # Mm³ the plant uses per hour at full load
        unit_per_hour = unit_max_flow[unit] * 3600 / 1e6

        hours_available = np.clip((vol_start + inflow_week) / plant_per_hour, 0, 168)
        hours_forced = np.clip((inflow_week - free_space) / plant_per_hour, 0, 168)
        unit_hours_available = np.clip((vol_start + inflow_week) / unit_per_hour, 0, 168)

        ups = plant_upstream[plant]
        if ups:
            up_diff = np.mean([diff_end_by_plant[q] for q in ups], axis=0)
            up_share = np.mean([(diff_end_by_plant[q] > 0).mean(axis=1) for q in ups], axis=0)
        else:
            up_diff = np.zeros_like(P)
            up_share = np.zeros(n_cases)

        blocks.append(pd.DataFrame({
            "case": rep(np.arange(n_cases)),
            "unit": unit,
            **shared,
            "wv_start": rep(wv_start),
            "wv_end": rep(wv_end),
            "net_wv_end": rep(net_wv_end),
            "price_minus_wv_end": diff_end.ravel(),
            "ratio_price_wv_end": (P / (np.abs(wv_end[:, None]) + 1)).ravel(),
            "share_above_wv_end": rep((diff_end > 0).mean(axis=1)),
            "roll6_minus_wv_end": rolling_within_case(diff_end, 7).ravel(),
            "roll24_minus_wv_end": rolling_within_case(diff_end, 25).ravel(),
            "above_wv_end_run": rolling_within_case((diff_end > 0).astype(float), 13).ravel(),
            "price_minus_net_end": diff_net.ravel(),
            "share_above_net_end": rep((diff_net > 0).mean(axis=1)),
            "fill": rep(daily_value(fill, starts, up)),
            "fill_down": rep(daily_value(fill, starts, down)),
            "inflow_res": res_inflow.ravel(),
            "inflow_res_week": rep(res_inflow.mean(axis=1)),
            "inflow_res_anom": rep(res_inflow.mean(axis=1) / (res_normal + 1e-3)),
            "upstream_roll6_minus_wv_end": rolling_within_case(up_diff, 7).ravel(),
            "upstream_share_above": rep(up_share),
            "hours_available": rep(hours_available),
            "hours_forced": rep(hours_forced),
            "unit_hours_available": rep(unit_hours_available),
            "rank_minus_available": (desc_rank - hours_available[:, None]).ravel(),
            "rank_minus_forced": (desc_rank - hours_forced[:, None]).ravel(),
        }))

    X = pd.concat(blocks, ignore_index=True)
    X["unit"] = pd.Categorical(X["unit"], categories=units)
    numeric = X.columns.difference(["case", "unit"])
    X[numeric] = X[numeric].astype("float32")         # saves memory
    return X

In [10]:
X = build_features(unit_commitment_decisions.index)
y = Y.transpose(1, 0, 2).ravel()      # same order as X: unit, then case, then hour

i, unit, h = 100, "Songa_G1", 37
row = X[(X["unit"] == unit) & (X["case"] == i) & (X["hour"] == h)].index[0]
assert y[row] == Y[i, units.index(unit), h]
X.shape                               # (6872544, 36)

(6872544, 36)

In [11]:
# Features for the network: the same as v6 LightGBM, but without "unit" and "case".
# The unit is known from its position in the array (0–13), so it doesn't need to be a feature.
absolute_levels = ["price", "price_week_mean", "price_roll6", "price_roll24", "wv_start", "wv_end"]
nn_features = [c for c in X.columns if c not in ["case", "unit"] + absolute_levels]

def to_weeks(X_rows, n_cases):
    """Rows ordered (unit, case, hour) -> array of shape (cases, units, hours, features)."""
    A = X_rows[nn_features].to_numpy(dtype=np.float32)
    A = A.reshape(len(units), n_cases, 168, len(nn_features))   # (unit, case, hour, feature)
    return A.transpose(1, 0, 2, 3)                               # (case, unit, hour, feature)

n_train_cases = len(unit_commitment_decisions)
A = to_weeks(X, n_train_cases)
Y_weeks = Y.astype(np.float32)                                    # already (case, unit, hour)

print(A.shape, Y_weeks.shape)   # (2922, 14, 168, 28) and (2922, 14, 168)

(2922, 14, 168, 28) (2922, 14, 168)


In [12]:
i, unit, h = 100, "Songa_G1", 37
k = units.index(unit)
row = X[(X["unit"] == unit) & (X["case"] == i) & (X["hour"] == h)].index[0]
assert np.allclose(A[i, k, h], X.loc[row, nn_features].to_numpy(dtype=np.float32))
assert Y_weeks[i, k, h] == y[row]
print("Week format OK")

Week format OK


In [13]:
case_years = unit_commitment_decisions.index.year.to_numpy()   # one year per case

def standardize(A, train_cases):
    """Scale each feature to mean 0 / std 1 using the training cases only, then clip extreme values."""
    mean = A[train_cases].mean(axis=(0, 1, 2), keepdims=True)
    std = A[train_cases].std(axis=(0, 1, 2), keepdims=True) + 1e-6
    return np.clip((A - mean) / std, -5, 5), mean, std

# The same validation periods as LightGBM
periods = {
    "2021–22": (case_years <= 2020, case_years >= 2021),
    "2022":    (case_years <= 2021, case_years == 2022),
    "2019–20": ((case_years <= 2018) | (case_years >= 2021), (case_years >= 2019) & (case_years <= 2020)),
}

tr, va = periods["2021–22"]
A_std, mean, std = standardize(A, tr)
print(tr.sum(), "training cases,", va.sum(), "validation cases")
print("Feature means after scaling (should be ~0):", A_std[tr].mean(axis=(0, 1, 2)).round(2)[:5])

2192 training cases, 730 validation cases
Feature means after scaling (should be ~0): [ 0.  0. -0. -0. -0.]


In [14]:
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)

class UnitCNN(nn.Module):
    """1D CNN over the 168 hours of a week, shared by all 14 units.
    Each unit gets a learned embedding, and every block also sees the mean of all units (system context).
    With week_context=True, every block also sees the mean over all 168 hours (the whole week)."""
    def __init__(self, n_features, n_units=14, emb_dim=8, hidden=64, dilations=(1, 2, 4, 8, 1), dropout=0.1, week_context=False):
        super().__init__()
        self.unit_emb = nn.Embedding(n_units, emb_dim)
        self.inp = nn.Conv1d(n_features + emb_dim, hidden, kernel_size=1)
        self.week_context = week_context
        n_in = (3 if week_context else 2) * hidden      # own state + unit mean (+ week mean)
        self.convs = nn.ModuleList([nn.Conv1d(n_in, hidden, kernel_size=3, padding=d, dilation=d) for d in dilations])
        self.norms = nn.ModuleList([nn.GroupNorm(8, hidden) for _ in dilations])
        self.drop = nn.Dropout(dropout)
        self.out = nn.Conv1d(hidden, 1, kernel_size=1)
        

    def forward(self, x):                      # x: (B, U, H, F)
        B, U, H, F = x.shape
        emb = self.unit_emb.weight[None, :, None, :].expand(B, U, H, -1)
        x = torch.cat([x, emb], dim=-1).reshape(B * U, H, -1).transpose(1, 2)   # (B*U, F+E, H)
        h = torch.relu(self.inp(x))
        for conv, norm in zip(self.convs, self.norms):
            ctx = h.reshape(B, U, -1, H).mean(dim=1, keepdim=True).expand(B, U, -1, H).reshape(B * U, -1, H)
            parts = [h, ctx]
            if self.week_context:
                parts.append(h.mean(dim=2, keepdim=True).expand(-1, -1, H))   # mean over all 168 hours
            h = h + self.drop(torch.relu(norm(conv(torch.cat(parts, dim=1)))))   # residual block
        return self.out(h).reshape(B, U, H)    # logits: (B, 14, 168)

print("Parameters:", sum(p.numel() for p in UnitCNN(len(nn_features)).parameters()))
print("Parameters with week context:", sum(p.numel() for p in UnitCNN(len(nn_features), week_context=True).parameters()))

Device: mps
Parameters: 126385
Parameters with week context: 187825


In [15]:
@torch.no_grad()
def predict_nn(model, X, batch_size=128):
    model.eval()
    out = [torch.sigmoid(model(X[i:i + batch_size].to(device))).cpu() for i in range(0, len(X), batch_size)]
    return torch.cat(out).numpy()

def train_nn(period, max_epochs=40, patience=5, batch_size=32, lr=3e-4, **model_kwargs):
    tr, va = periods[period]
    A_s, _, _ = standardize(A, tr)                       # scaled on this period's training cases only
    X_tr, Y_tr = torch.from_numpy(A_s[tr]), torch.from_numpy(Y_weeks[tr])
    X_va, y_va = torch.from_numpy(A_s[va]), Y_weeks[va].ravel()

    torch.manual_seed(42)
    model = UnitCNN(len(nn_features), **model_kwargs).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()
    best_auc, best_epoch, best_state, bad, best_p = 0.0, 0, None, 0, None

    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        model.train()
        order = torch.randperm(len(X_tr))
        total = 0.0
        for i in range(0, len(order), batch_size):
            idx = order[i:i + batch_size]
            xb, yb = X_tr[idx].to(device), Y_tr[idx].to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item() * len(idx)

        p_va = predict_nn(model, X_va)
        auc = roc_auc_score(y_va, p_va.ravel())
        print(f"{period} | epoch {epoch:2d} | loss {total / len(X_tr):.4f} | val AUC {auc:.4f} | {time.time() - t0:.0f}s")
        if auc > best_auc + 1e-4:
            best_auc, best_epoch, bad, best_p = auc, epoch, 0, p_va
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_auc, best_epoch, best_p

In [16]:
del A_std   # free the copy from step 1; train_nn makes its own

In [17]:
# Validation: NN alone, and rank blends with two LightGBM bases (v6, and the v5/v6 average from 02)
results = []
for period in periods:
    tr, va = periods[period]
    key = period.replace("–", "_")
    y_va = Y_weeks[va].ravel()

    _, _, best_ep, p_nn = train_nn(period, max_epochs=40, lr=3e-4)
    p_nn = p_nn.ravel()
    np.save(f"nn_valid_{key}.npy", p_nn)                  # saved, so later blend tests need no training
    r_nn = rankdata(p_nn) / len(p_nn)

    row = {"period": period, "NN": roc_auc_score(y_va, p_nn), "NN epoch": best_ep}
    for base, fname in [("v6", f"lgb_valid_{key}.npy"), ("avg", f"lgb_avg_valid_{key}.npy")]:
        p_lgb = np.load(fname).ravel()
        r_lgb = rankdata(p_lgb) / len(p_lgb)
        row[f"LGBM {base}"] = roc_auc_score(y_va, p_lgb)
        for w in [0.8, 0.7, 0.6]:
            row[f"{base}+NN w={w}"] = roc_auc_score(y_va, w * r_lgb + (1 - w) * r_nn)
    results.append(row)

results = pd.DataFrame(results).set_index("period").round(4)
results.loc["mean"] = results.mean().round(4)
results.T

2021–22 | epoch  1 | loss 0.3917 | val AUC 0.9426 | 10s
2021–22 | epoch  2 | loss 0.2696 | val AUC 0.9532 | 10s
2021–22 | epoch  3 | loss 0.2248 | val AUC 0.9566 | 10s
2021–22 | epoch  4 | loss 0.2079 | val AUC 0.9553 | 10s
2021–22 | epoch  5 | loss 0.1963 | val AUC 0.9571 | 10s
2021–22 | epoch  6 | loss 0.1836 | val AUC 0.9584 | 9s
2021–22 | epoch  7 | loss 0.1724 | val AUC 0.9562 | 9s
2021–22 | epoch  8 | loss 0.1692 | val AUC 0.9596 | 10s
2021–22 | epoch  9 | loss 0.1645 | val AUC 0.9567 | 9s
2021–22 | epoch 10 | loss 0.1580 | val AUC 0.9582 | 10s
2021–22 | epoch 11 | loss 0.1539 | val AUC 0.9601 | 10s
2021–22 | epoch 12 | loss 0.1534 | val AUC 0.9611 | 9s
2021–22 | epoch 13 | loss 0.1465 | val AUC 0.9593 | 10s
2021–22 | epoch 14 | loss 0.1464 | val AUC 0.9591 | 10s
2021–22 | epoch 15 | loss 0.1416 | val AUC 0.9591 | 9s
2021–22 | epoch 16 | loss 0.1424 | val AUC 0.9589 | 10s
2021–22 | epoch 17 | loss 0.1362 | val AUC 0.9592 | 9s
2022 | epoch  1 | loss 0.3854 | val AUC 0.9516 | 10s
2

period,2021–22,2022,2019–20,mean
NN,0.9611,0.9578,0.9621,0.9603
NN epoch,12.0000,3.0000,23.0000,12.6667
LGBM v6,0.9685,0.9765,0.9578,0.9676
v6+NN w=0.8,0.9717,0.9778,0.9627,0.9707
v6+NN w=0.7,0.9724,0.9774,0.9643,0.9714
v6+NN w=0.6,0.9725,0.9764,0.9655,0.9715
LGBM avg,0.9682,0.9764,0.9596,0.9681
avg+NN w=0.8,0.9714,0.9775,0.9640,0.9710
avg+NN w=0.7,0.9721,0.9771,0.9654,0.9715
avg+NN w=0.6,0.9722,0.9762,0.9664,0.9716


## Kaggle test: NN trained on all data, blended with LightGBM v6

Validation (table above): rank average with w_lgb = 0.7 beat LightGBM alone in all three periods.
To save time, skip the validation loop above: run the cells up to and including `del A_std`, then the cells below.
The LightGBM predictions are read from `submission_lgbm_v6.csv` (made in 02), so LightGBM is not imported here.

In [18]:
# Test data in week format, scaled with mean/std from ALL training cases
X_test = build_features(test_cases["starttime"])
A_test = to_weeks(X_test, len(test_cases))

all_cases = np.ones(len(A), dtype=bool)
A_all, mean_all, std_all = standardize(A, all_cases)
A_test_s = np.clip((A_test - mean_all) / std_all, -5, 5).astype(np.float32)
print(A_all.shape, A_test_s.shape)

(2922, 14, 168, 28) (725, 14, 168, 28)


In [25]:
# Train on all cases for a fixed number of epochs (no validation set to stop on)
def train_final(A_s, epochs=12, batch_size=32, lr=3e-4, **model_kwargs):
    X_tr, Y_tr = torch.from_numpy(A_s), torch.from_numpy(Y_weeks)
    torch.manual_seed(42)
    model = UnitCNN(len(nn_features), **model_kwargs).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        order = torch.randperm(len(X_tr))
        total = 0.0
        for i in range(0, len(order), batch_size):
            idx = order[i:i + batch_size]
            xb, yb = X_tr[idx].to(device), Y_tr[idx].to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item() * len(idx)
        print(f"epoch {epoch:2d} | loss {total / len(X_tr):.4f} | {time.time() - t0:.0f}s")
    return model

In [2]:
# Original NN on all data (12 epochs) and its test predictions
final_nn = train_final(A_all)
p_test_nn = predict_nn(final_nn, torch.from_numpy(A_test_s))     # (test case, unit, hour)
p_test_nn = p_test_nn.reshape(len(test_cases), -1)               # same column order as label_cols

NameError: name 'train_final' is not defined

In [20]:
# Rank-average blend with LightGBM v6 and save
version = "v8_nn_blend_avg"
w_lgb = 0.7

lgb_sub = pd.read_csv("submission_lgbm_v7_avg.csv")
assert list(lgb_sub.columns[1:]) == label_cols and len(lgb_sub) == len(test_cases)
assert (lgb_sub["Run No"].to_numpy() == test_cases["Run No"].to_numpy()).all()
p_test_lgb = lgb_sub[label_cols].to_numpy()

r_lgb = rankdata(p_test_lgb.ravel()) / p_test_lgb.size
r_nn = rankdata(p_test_nn.ravel()) / p_test_nn.size
print("Rank correlation LGBM vs NN on test (should be high, e.g. > 0.9):", round(np.corrcoef(r_lgb, r_nn)[0, 1], 3))

submission = lgb_sub.copy()
submission[label_cols] = (w_lgb * r_lgb + (1 - w_lgb) * r_nn).reshape(p_test_lgb.shape)
submission.to_csv(f"submission_{version}.csv", index=False)
submission.shape   # (725, 2353)

Rank correlation LGBM vs NN on test (should be high, e.g. > 0.9): 0.951


(725, 2353)

## NN improvement A: week context

The original CNN only sees about ±16 hours around each hour (kernel 3, dilations 1, 2, 4, 8, 1 → receptive field 33 hours).
With `week_context=True`, every block also gets the mean over all 168 hours, so each hour "sees" the whole week.
Needs: cells up to and including `del A_std`, plus `nn_valid_*.npy` and `lgb_avg_valid_*.npy` (saved earlier). About 10 minutes.

In [23]:
r = lambda p: rankdata(p) / len(p)      # rank-normalise for blending

rows = {}
for period in periods:
    tr, va = periods[period]
    key = period.replace("–", "_")
    y_va = Y_weeks[va].ravel()

    _, _, best_ep, p_wc = train_nn(period, max_epochs=40, lr=3e-4, week_context=True)
    p_wc = p_wc.ravel()
    np.save(f"nn_wc_valid_{key}.npy", p_wc)

    p_old = np.load(f"nn_valid_{key}.npy")                    # original NN (saved by the validation loop above)
    p_avg = np.load(f"lgb_avg_valid_{key}.npy").ravel()       # LightGBM v5/v6 average (saved in 02)

    rows[period] = {
        "NN (original)":            roc_auc_score(y_va, p_old),
        "NN week context":          roc_auc_score(y_va, p_wc),
        "week context best epoch":  best_ep,
        "avg + original  w=0.7":    roc_auc_score(y_va, 0.7 * r(p_avg) + 0.3 * r(p_old)),
        "avg + week ctx  w=0.7":    roc_auc_score(y_va, 0.7 * r(p_avg) + 0.3 * r(p_wc)),
        "avg + week ctx  w=0.6":    roc_auc_score(y_va, 0.6 * r(p_avg) + 0.4 * r(p_wc)),
        "avg + both NNs  0.6/0.2/0.2": roc_auc_score(y_va, 0.6 * r(p_avg) + 0.2 * r(p_old) + 0.2 * r(p_wc)),
    }

wc_table = pd.DataFrame(rows)
wc_table["mean"] = wc_table.mean(axis=1)
wc_table.round(4)

2021–22 | epoch  1 | loss 0.3631 | val AUC 0.9436 | 14s
2021–22 | epoch  2 | loss 0.2553 | val AUC 0.9527 | 13s
2021–22 | epoch  3 | loss 0.2252 | val AUC 0.9571 | 13s
2021–22 | epoch  4 | loss 0.2031 | val AUC 0.9591 | 13s
2021–22 | epoch  5 | loss 0.2019 | val AUC 0.9612 | 13s
2021–22 | epoch  6 | loss 0.1887 | val AUC 0.9597 | 13s
2021–22 | epoch  7 | loss 0.1756 | val AUC 0.9593 | 13s
2021–22 | epoch  8 | loss 0.1682 | val AUC 0.9604 | 13s
2021–22 | epoch  9 | loss 0.1637 | val AUC 0.9605 | 13s
2021–22 | epoch 10 | loss 0.1612 | val AUC 0.9599 | 13s
2022 | epoch  1 | loss 0.3641 | val AUC 0.9517 | 14s
2022 | epoch  2 | loss 0.2469 | val AUC 0.9600 | 14s
2022 | epoch  3 | loss 0.2205 | val AUC 0.9589 | 14s
2022 | epoch  4 | loss 0.2023 | val AUC 0.9626 | 14s
2022 | epoch  5 | loss 0.1955 | val AUC 0.9635 | 14s
2022 | epoch  6 | loss 0.1851 | val AUC 0.9629 | 14s
2022 | epoch  7 | loss 0.1832 | val AUC 0.9661 | 14s
2022 | epoch  8 | loss 0.1705 | val AUC 0.9624 | 14s
2022 | epoch  9 

,2021–22,2022,2019–20,mean
NN (original),0.9611,0.9578,0.9621,0.9603
NN week context,0.9612,0.9661,0.9607,0.9626
week context best epoch,5.0000,7.0000,12.0000,8.0000
avg + original w=0.7,0.9721,0.9771,0.9654,0.9716
avg + week ctx w=0.7,0.9720,0.9785,0.9653,0.9719
avg + week ctx w=0.6,0.9722,0.9783,0.9661,0.9722
avg + both NNs 0.6/0.2/0.2,0.9726,0.9777,0.9666,0.9723


## Kaggle: v5/v6 average + original NN + week-context NN (0.6 / 0.2 / 0.2)

Best blend in the table above: beats `avg + original w=0.7` (current Kaggle best, 0.98437) in all three periods.
The week-context NN is trained on all data for 8 epochs (its best epochs on validation were 5, 7 and 12).
Needs: cells up to and including `del A_std`, and the "Test data in week format" cell (`A_all`, `A_test_s`).
If the original NN's test predictions (`p_test_nn`) are no longer in memory, it is trained again first (12 epochs).

In [26]:
version = "v9_two_nn"
weights = {"lgb": 0.6, "nn": 0.2, "nn_wc": 0.2}

if "p_test_nn" not in globals():                                   # original NN, as in v8
    p_test_nn = predict_nn(train_final(A_all, epochs=12), torch.from_numpy(A_test_s)).reshape(len(test_cases), -1)

final_nn_wc = train_final(A_all, epochs=8, week_context=True)
p_test_wc = predict_nn(final_nn_wc, torch.from_numpy(A_test_s)).reshape(len(test_cases), -1)

lgb_sub = pd.read_csv("submission_lgbm_v7_avg.csv")              # LightGBM v5/v6 average (made in 02)
assert list(lgb_sub.columns[1:]) == label_cols and len(lgb_sub) == len(test_cases)
assert (lgb_sub["Run No"].to_numpy() == test_cases["Run No"].to_numpy()).all()
p_test_lgb = lgb_sub[label_cols].to_numpy()

rr = lambda p: rankdata(p.ravel()) / p.size
r_lgb, r_nn, r_wc = rr(p_test_lgb), rr(p_test_nn), rr(p_test_wc)
print("Rank correlations on test (should all be high):",
      "LGBM–NN", round(np.corrcoef(r_lgb, r_nn)[0, 1], 3),
      "| LGBM–NN wc", round(np.corrcoef(r_lgb, r_wc)[0, 1], 3),
      "| NN–NN wc", round(np.corrcoef(r_nn, r_wc)[0, 1], 3))

blend = weights["lgb"] * r_lgb + weights["nn"] * r_nn + weights["nn_wc"] * r_wc
submission = lgb_sub.copy()
submission[label_cols] = blend.reshape(p_test_lgb.shape)
submission.to_csv(f"submission_{version}.csv", index=False)
submission.shape   # (725, 2353)

epoch  1 | loss 0.3530 | 16s
epoch  2 | loss 0.2509 | 16s
epoch  3 | loss 0.2272 | 15s
epoch  4 | loss 0.2139 | 16s
epoch  5 | loss 0.2034 | 15s
epoch  6 | loss 0.1983 | 15s
epoch  7 | loss 0.1903 | 15s
epoch  8 | loss 0.1807 | 15s
Rank correlations on test (should all be high): LGBM–NN 0.951 | LGBM–NN wc 0.952 | NN–NN wc 0.987


(725, 2353)

## NN improvement: long range (third NN)

Dilations (1, 2, 4, 8, 16, 32, 64, 1) give a receptive field of about 255 hours, so every hour can see the whole 168-hour week directly
(the original dilations only reach ±16 hours). Week context is kept on.
Needs: cells up to and including `del A_std` (with the updated model and `train_nn` cells), plus the saved
`nn_valid_*`, `nn_wc_valid_*` and `lgb_avg_valid_*` files. About 15 minutes (more layers, so slower epochs).

In [27]:
r = lambda p: rankdata(p) / len(p)      # rank-normalise for blending
long_range = (1, 2, 4, 8, 16, 32, 64, 1)

rows = {}
for period in periods:
    tr, va = periods[period]
    key = period.replace("–", "_")
    y_va = Y_weeks[va].ravel()

    _, _, best_ep, p_rf = train_nn(period, max_epochs=40, lr=3e-4, week_context=True, dilations=long_range)
    p_rf = p_rf.ravel()
    np.save(f"nn_rf_valid_{key}.npy", p_rf)

    p_old = np.load(f"nn_valid_{key}.npy")                    # original NN
    p_wc = np.load(f"nn_wc_valid_{key}.npy")                  # week-context NN
    p_avg = np.load(f"lgb_avg_valid_{key}.npy").ravel()       # LightGBM v5/v6 average
    a, o, w, f = r(p_avg), r(p_old), r(p_wc), r(p_rf)

    rows[period] = {
        "NN week context":                    roc_auc_score(y_va, p_wc),
        "NN long range":                      roc_auc_score(y_va, p_rf),
        "long range best epoch":              best_ep,
        "avg + orig + wc   0.6/0.2/0.2 (v9)": roc_auc_score(y_va, 0.6 * a + 0.2 * o + 0.2 * w),
        "avg + orig + rf   0.6/0.2/0.2":      roc_auc_score(y_va, 0.6 * a + 0.2 * o + 0.2 * f),
        "avg + wc + rf     0.6/0.2/0.2":      roc_auc_score(y_va, 0.6 * a + 0.2 * w + 0.2 * f),
        "avg + 3 NNs       0.55/0.15x3":      roc_auc_score(y_va, 0.55 * a + 0.15 * (o + w + f)),
        "avg + 3 NNs       0.5/0.167x3":      roc_auc_score(y_va, 0.5 * a + (o + w + f) / 6),
    }

rf_table = pd.DataFrame(rows)
rf_table["mean"] = rf_table.mean(axis=1)
rf_table.round(4)

2021–22 | epoch  1 | loss 0.4135 | val AUC 0.9405 | 20s
2021–22 | epoch  2 | loss 0.2652 | val AUC 0.9506 | 20s
2021–22 | epoch  3 | loss 0.2187 | val AUC 0.9526 | 20s
2021–22 | epoch  4 | loss 0.2000 | val AUC 0.9569 | 20s
2021–22 | epoch  5 | loss 0.1928 | val AUC 0.9549 | 20s
2021–22 | epoch  6 | loss 0.1823 | val AUC 0.9577 | 20s
2021–22 | epoch  7 | loss 0.1767 | val AUC 0.9581 | 20s
2021–22 | epoch  8 | loss 0.1725 | val AUC 0.9576 | 20s
2021–22 | epoch  9 | loss 0.1638 | val AUC 0.9604 | 20s
2021–22 | epoch 10 | loss 0.1634 | val AUC 0.9570 | 20s
2021–22 | epoch 11 | loss 0.1560 | val AUC 0.9581 | 20s
2021–22 | epoch 12 | loss 0.1520 | val AUC 0.9549 | 20s
2021–22 | epoch 13 | loss 0.1538 | val AUC 0.9614 | 20s
2021–22 | epoch 14 | loss 0.1457 | val AUC 0.9609 | 20s
2021–22 | epoch 15 | loss 0.1436 | val AUC 0.9599 | 21s
2021–22 | epoch 16 | loss 0.1421 | val AUC 0.9596 | 22s
2021–22 | epoch 17 | loss 0.1420 | val AUC 0.9582 | 24s
2021–22 | epoch 18 | loss 0.1361 | val AUC 0.960

,2021–22,2022,2019–20,mean
NN week context,0.9612,0.9661,0.9607,0.9626
NN long range,0.9614,0.9656,0.9638,0.9636
long range best epoch,13.0000,15.0000,15.0000,14.3333
avg + orig + wc 0.6/0.2/0.2 (v9),0.9726,0.9777,0.9666,0.9723
avg + orig + rf 0.6/0.2/0.2,0.9732,0.9783,0.9668,0.9728
avg + wc + rf 0.6/0.2/0.2,0.9731,0.9788,0.9666,0.9729
avg + 3 NNs 0.55/0.15x3,0.9732,0.9782,0.9672,0.9729
avg + 3 NNs 0.5/0.167x3,0.9731,0.9779,0.9675,0.9728


## Kaggle: v5/v6 average + three NNs (0.55 / 0.15 / 0.15 / 0.15)

Beats v9 (Kaggle 0.98504) in all three validation periods: 0.9732 / 0.9782 / 0.9672 vs 0.9726 / 0.9777 / 0.9666.
The long-range NN is trained on all data for 14 epochs (its best epochs on validation were 13, 15 and 15).
Test predictions of the original and week-context NNs are reused if still in memory, otherwise trained again.

In [28]:
version = "v10_three_nn"
long_range = (1, 2, 4, 8, 16, 32, 64, 1)
to_test = lambda m: predict_nn(m, torch.from_numpy(A_test_s)).reshape(len(test_cases), -1)

if "p_test_nn" not in globals():                                   # original NN (12 epochs)
    p_test_nn = to_test(train_final(A_all, epochs=12))
if "p_test_wc" not in globals():                                   # week-context NN (8 epochs)
    p_test_wc = to_test(train_final(A_all, epochs=8, week_context=True))
p_test_rf = to_test(train_final(A_all, epochs=14, week_context=True, dilations=long_range))   # long-range NN

lgb_sub = pd.read_csv("submission_lgbm_v7_avg.csv")              # LightGBM v5/v6 average (made in 02)
assert list(lgb_sub.columns[1:]) == label_cols and len(lgb_sub) == len(test_cases)
assert (lgb_sub["Run No"].to_numpy() == test_cases["Run No"].to_numpy()).all()
p_test_lgb = lgb_sub[label_cols].to_numpy()

rr = lambda p: rankdata(p.ravel()) / p.size
r_lgb, r_nn, r_wc, r_rf = rr(p_test_lgb), rr(p_test_nn), rr(p_test_wc), rr(p_test_rf)
print("Rank correlations on test:",
      "LGBM–long range", round(np.corrcoef(r_lgb, r_rf)[0, 1], 3),
      "| week ctx–long range", round(np.corrcoef(r_wc, r_rf)[0, 1], 3))

blend = 0.55 * r_lgb + 0.15 * (r_nn + r_wc + r_rf)
submission = lgb_sub.copy()
submission[label_cols] = blend.reshape(p_test_lgb.shape)
submission.to_csv(f"submission_{version}.csv", index=False)
submission.shape   # (725, 2353)

epoch  1 | loss 0.3816 | 24s
epoch  2 | loss 0.2626 | 23s
epoch  3 | loss 0.2286 | 23s
epoch  4 | loss 0.2213 | 23s
epoch  5 | loss 0.2144 | 23s
epoch  6 | loss 0.1983 | 23s
epoch  7 | loss 0.1893 | 23s
epoch  8 | loss 0.1832 | 26s
epoch  9 | loss 0.1756 | 33s
epoch 10 | loss 0.1698 | 30s
epoch 11 | loss 0.1675 | 29s
epoch 12 | loss 0.1608 | 28s
epoch 13 | loss 0.1665 | 28s
epoch 14 | loss 0.1564 | 28s
Rank correlations on test: LGBM–long range 0.946 | week ctx–long range 0.984


(725, 2353)

## Save the three NNs' test predictions (for the XGBoost blend in 02)

Reuses the predictions in memory if they are there (from v10); otherwise trains the NNs again on all data first
(then it needs the cells up to `del A_std`, the "Test data in week format" cell and the `train_final` cell; about 12 minutes).

In [1]:
to_test = lambda m: predict_nn(m, torch.from_numpy(A_test_s)).reshape(len(test_cases), -1)   # Kaggle layout
long_range = (1, 2, 4, 8, 16, 32, 64, 1)

if "p_test_nn" not in globals():
    p_test_nn = to_test(train_final(A_all, epochs=12))
if "p_test_wc" not in globals():
    p_test_wc = to_test(train_final(A_all, epochs=8, week_context=True))
if "p_test_rf" not in globals():
    p_test_rf = to_test(train_final(A_all, epochs=14, week_context=True, dilations=long_range))

np.save("nn_test_orig.npy", p_test_nn)
np.save("nn_test_wc.npy", p_test_wc)
np.save("nn_test_rf.npy", p_test_rf)
print(p_test_nn.shape, p_test_wc.shape, p_test_rf.shape)   # each (725, 2352)

NameError: name 'train_final' is not defined

## NN with system features (round 3, step 4)

The per-unit XGBoost gained most from the system features (the state of all 17 reservoirs), so the NN gets them too:
for each hour, 68 values (price minus end water value, end water value, start fill and weekly inflow for every reservoir;
the price/water-value columns go through a signed log). They are projected to the hidden size and added to every unit's hidden state.
Architecture otherwise as the long-range NN (week context on, dilations up to 64).

Run: the cells up to and including `del A_std`, then the cells below. Validation saves `nn_sys_valid_*.npy` (about 15–20 minutes);
the final cell trains on all data with 3 seeds and saves `nn_sys_test.npy` (about 15 minutes). Then go to step 5 in 02.

In [17]:
# System features: the state of all 17 reservoirs, one row per (case, hour) -> array (cases, 168, 68)
import json
reservoirs = list(water_value.columns)

def system_features(starts):
    """Same as in 02."""
    starts = pd.DatetimeIndex(starts)
    ends = starts + pd.Timedelta(days=7)
    P = hourly_matrix(day_ahead_price["price"], starts)                                   # (cases, 168)
    wv_end = water_value.loc[ends, reservoirs].to_numpy()                                 # (cases, 17)
    fill_start = fill.loc[starts, reservoirs].to_numpy()
    inflow_week = np.stack([hourly_matrix(inflow[r], starts).mean(axis=1) for r in reservoirs], axis=1)
    rep = lambda v: np.repeat(v, 168)
    cols = {}
    for j, r in enumerate(reservoirs):
        cols[f"sys_margin_{r}"] = (P - wv_end[:, j][:, None]).ravel()
    for j, r in enumerate(reservoirs):
        cols[f"sys_wv_end_{r}"] = rep(wv_end[:, j])
        cols[f"sys_fill_{r}"] = rep(fill_start[:, j])
        cols[f"sys_inflow_{r}"] = rep(inflow_week[:, j])
    return pd.DataFrame(cols).astype("float32")

def sys_array(starts):
    """(cases, 168, 68); signed log on the price/water-value columns so the 2022 price spikes do not dominate."""
    D = system_features(starts)
    for c in D.columns:
        if c.startswith(("sys_margin_", "sys_wv_end_")):
            v = D[c].to_numpy()
            D[c] = np.sign(v) * np.log1p(np.abs(v))
    return D.to_numpy(dtype=np.float32).reshape(len(starts), 168, D.shape[1])

def standardize_sys(S, train_cases):
    """Mean 0 / std 1 per system feature, using the training cases only, then clip."""
    mean = S[train_cases].mean(axis=(0, 1), keepdims=True)
    std = S[train_cases].std(axis=(0, 1), keepdims=True) + 1e-6
    return np.clip((S - mean) / std, -5, 5).astype(np.float32), mean, std

S = sys_array(unit_commitment_decisions.index)
print(S.shape, "- no NaN:", not np.isnan(S).any())   # (2922, 168, 68)

(2922, 168, 68) - no NaN: True


In [18]:
class SysUnitCNN(nn.Module):
    """UnitCNN + system features: the 68 system values per hour are projected to the hidden size
    and added to every unit's hidden state after the input layer."""
    def __init__(self, n_features, n_sys, n_units=14, emb_dim=8, hidden=64, dilations=(1, 2, 4, 8, 1), dropout=0.1, week_context=False):
        super().__init__()
        self.unit_emb = nn.Embedding(n_units, emb_dim)
        self.inp = nn.Conv1d(n_features + emb_dim, hidden, kernel_size=1)
        self.sys_proj = nn.Conv1d(n_sys, hidden, kernel_size=1)
        self.week_context = week_context
        n_in = (3 if week_context else 2) * hidden
        self.convs = nn.ModuleList([nn.Conv1d(n_in, hidden, kernel_size=3, padding=d, dilation=d) for d in dilations])
        self.norms = nn.ModuleList([nn.GroupNorm(8, hidden) for _ in dilations])
        self.drop = nn.Dropout(dropout)
        self.out = nn.Conv1d(hidden, 1, kernel_size=1)

    def forward(self, x, s):                   # x: (B, U, H, F), s: (B, H, S)
        B, U, H, F = x.shape
        emb = self.unit_emb.weight[None, :, None, :].expand(B, U, H, -1)
        x = torch.cat([x, emb], dim=-1).reshape(B * U, H, -1).transpose(1, 2)    # (B*U, F+E, H)
        s = self.sys_proj(s.transpose(1, 2))                                      # (B, hidden, H)
        s = s[:, None].expand(B, U, -1, H).reshape(B * U, -1, H)                  # same for all 14 units
        h = torch.relu(self.inp(x) + s)
        for conv, norm in zip(self.convs, self.norms):
            ctx = h.reshape(B, U, -1, H).mean(dim=1, keepdim=True).expand(B, U, -1, H).reshape(B * U, -1, H)
            parts = [h, ctx]
            if self.week_context:
                parts.append(h.mean(dim=2, keepdim=True).expand(-1, -1, H))
            h = h + self.drop(torch.relu(norm(conv(torch.cat(parts, dim=1)))))
        return self.out(h).reshape(B, U, H)

@torch.no_grad()
def predict_sys(model, X, S_, batch_size=128):
    model.eval()
    out = [torch.sigmoid(model(X[i:i + batch_size].to(device), S_[i:i + batch_size].to(device))).cpu()
           for i in range(0, len(X), batch_size)]
    return torch.cat(out).numpy()

def fit_sys(A_s, S_s, train, valid=None, epochs=40, patience=5, batch_size=32, lr=3e-4, seed=42, **model_kwargs):
    """Train on the cases in `train`. With `valid`, stop early on its AUC and keep the best epoch;
    without, train for exactly `epochs` epochs (final model)."""
    X_tr, S_tr, Y_tr = torch.from_numpy(A_s[train]), torch.from_numpy(S_s[train]), torch.from_numpy(Y_weeks[train])
    if valid is not None:
        X_va, S_va, y_va = torch.from_numpy(A_s[valid]), torch.from_numpy(S_s[valid]), Y_weeks[valid].ravel()
    torch.manual_seed(seed)
    model = SysUnitCNN(len(nn_features), S_s.shape[-1], **model_kwargs).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()
    best_auc, best_epoch, best_state, bad, best_p = 0.0, 0, None, 0, None

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        order = torch.randperm(len(X_tr))
        total = 0.0
        for i in range(0, len(order), batch_size):
            idx = order[i:i + batch_size]
            opt.zero_grad()
            loss = loss_fn(model(X_tr[idx].to(device), S_tr[idx].to(device)), Y_tr[idx].to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item() * len(idx)
        msg = f"seed {seed} | epoch {epoch:2d} | loss {total / len(X_tr):.4f}"
        if valid is not None:
            p_va = predict_sys(model, X_va, S_va)
            auc = roc_auc_score(y_va, p_va.ravel())
            msg += f" | val AUC {auc:.4f}"
            if auc > best_auc + 1e-4:
                best_auc, best_epoch, bad, best_p = auc, epoch, 0, p_va
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
        print(msg + f" | {time.time() - t0:.0f}s")
        if valid is not None and bad >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_auc, best_epoch, best_p

sys_kwargs = dict(week_context=True, dilations=(1, 2, 4, 8, 16, 32, 64, 1))   # as the long-range NN
print("Parameters:", sum(p.numel() for p in SysUnitCNN(len(nn_features), S.shape[-1], **sys_kwargs).parameters()))

Parameters: 303409


In [19]:
# Validation in the three periods (about 15–20 minutes); saves nn_sys_valid_*.npy for the blend sweep in 02
sys_rows, sys_epochs = {}, {}
for period, (tr, va) in periods.items():
    key = period.replace("–", "_")
    A_s, _, _ = standardize(A, tr)                 # scaled on this period's training cases only
    S_s, _, _ = standardize_sys(S, tr)
    _, auc, ep, p = fit_sys(A_s, S_s, tr, va, **sys_kwargs)
    del A_s, S_s
    np.save(f"nn_sys_valid_{key}.npy", p.ravel())  # (case, unit, hour) order, same as Y[va].ravel() in 02
    sys_epochs[period] = ep
    y_va = Y_weeks[va].ravel()
    sys_rows[period] = {"NN long range (no system features)": roc_auc_score(y_va, np.load(f"nn_rf_valid_{key}.npy")),
                        "NN + system features":               auc,
                        "best epoch":                         ep}

json.dump(sys_epochs, open("nn_sys_epochs.json", "w"))
sys_table = pd.DataFrame(sys_rows)
sys_table["mean"] = sys_table.mean(axis=1)
sys_table.round(4)

seed 42 | epoch  1 | loss 0.4182 | val AUC 0.9338 | 21s
seed 42 | epoch  2 | loss 0.2445 | val AUC 0.9481 | 19s
seed 42 | epoch  3 | loss 0.2032 | val AUC 0.9519 | 19s
seed 42 | epoch  4 | loss 0.1867 | val AUC 0.9514 | 19s
seed 42 | epoch  5 | loss 0.1755 | val AUC 0.9555 | 19s
seed 42 | epoch  6 | loss 0.1645 | val AUC 0.9571 | 19s
seed 42 | epoch  7 | loss 0.1661 | val AUC 0.9562 | 19s
seed 42 | epoch  8 | loss 0.1525 | val AUC 0.9570 | 19s
seed 42 | epoch  9 | loss 0.1458 | val AUC 0.9576 | 19s
seed 42 | epoch 10 | loss 0.1429 | val AUC 0.9566 | 19s
seed 42 | epoch 11 | loss 0.1386 | val AUC 0.9573 | 19s
seed 42 | epoch 12 | loss 0.1382 | val AUC 0.9570 | 19s
seed 42 | epoch 13 | loss 0.1342 | val AUC 0.9566 | 19s
seed 42 | epoch 14 | loss 0.1292 | val AUC 0.9589 | 19s
seed 42 | epoch 15 | loss 0.1254 | val AUC 0.9613 | 19s
seed 42 | epoch 16 | loss 0.1235 | val AUC 0.9573 | 20s
seed 42 | epoch 17 | loss 0.1228 | val AUC 0.9580 | 21s
seed 42 | epoch 18 | loss 0.1196 | val AUC 0.960

,2021–22,2022,2019–20,mean
NN long range (no system features),0.9614,0.9656,0.9638,0.9636
NN + system features,0.9613,0.9723,0.9580,0.9639
best epoch,15.0000,15.0000,6.0000,12.0000


In [20]:
# Final: all training cases, epochs = median best epoch from validation, 3 seeds -> nn_sys_test.npy (Kaggle layout)
seeds = (42, 43, 44)                                   # (42,) for one seed, about a third of the time
try:
    epochs_final = int(np.median(list(json.load(open("nn_sys_epochs.json")).values())))
except FileNotFoundError:
    epochs_final = 14                                  # as the long-range NN, if validation was skipped
print("Epochs:", epochs_final)

if "A_test_s" not in globals():                        # same as the "Test data in week format" cell
    X_test = build_features(test_cases["starttime"])
    A_test = to_weeks(X_test, len(test_cases))
    del X_test
    A_all, mean_all, std_all = standardize(A, np.ones(len(A), dtype=bool))
    A_test_s = np.clip((A_test - mean_all) / std_all, -5, 5).astype(np.float32)

S_all, mean_s, std_s = standardize_sys(S, np.ones(len(S), dtype=bool))
S_test_s = np.clip((sys_array(test_cases["starttime"]) - mean_s) / std_s, -5, 5).astype(np.float32)

preds = []
for seed in seeds:
    m, *_ = fit_sys(A_all, S_all, slice(None), None, epochs=epochs_final, seed=seed, **sys_kwargs)   # slice: no copy
    preds.append(predict_sys(m, torch.from_numpy(A_test_s), torch.from_numpy(S_test_s)).reshape(len(test_cases), -1))
p_test_sys = np.mean(preds, axis=0)
np.save("nn_sys_test.npy", p_test_sys)
print(p_test_sys.shape)   # (725, 2352)

Epochs: 15
seed 42 | epoch  1 | loss 0.3791 | 25s
seed 42 | epoch  2 | loss 0.2329 | 24s
seed 42 | epoch  3 | loss 0.2061 | 24s
seed 42 | epoch  4 | loss 0.1846 | 23s
seed 42 | epoch  5 | loss 0.1719 | 24s
seed 42 | epoch  6 | loss 0.1655 | 24s
seed 42 | epoch  7 | loss 0.1595 | 23s
seed 42 | epoch  8 | loss 0.1517 | 23s
seed 42 | epoch  9 | loss 0.1515 | 23s
seed 42 | epoch 10 | loss 0.1447 | 24s
seed 42 | epoch 11 | loss 0.1386 | 25s
seed 42 | epoch 12 | loss 0.1367 | 28s
seed 42 | epoch 13 | loss 0.1324 | 30s
seed 42 | epoch 14 | loss 0.1347 | 29s
seed 42 | epoch 15 | loss 0.1308 | 30s
seed 43 | epoch  1 | loss 0.3937 | 32s
seed 43 | epoch  2 | loss 0.2338 | 26s
seed 43 | epoch  3 | loss 0.2009 | 25s
seed 43 | epoch  4 | loss 0.1833 | 26s
seed 43 | epoch  5 | loss 0.1746 | 26s
seed 43 | epoch  6 | loss 0.1601 | 27s
seed 43 | epoch  7 | loss 0.1558 | 27s
seed 43 | epoch  8 | loss 0.1486 | 26s
seed 43 | epoch  9 | loss 0.1448 | 26s
seed 43 | epoch 10 | loss 0.1403 | 26s
seed 43 | epoc